# Preparação dos dados do Ensembl

Este notebook existe para preparar os arquivos mais recentes do Ensembl antes de qualquer etapa de treino ou avaliação.

Os dados vêm compactados, com nomes longos e sem padronização direta para uso no pipeline. Se usados assim, isso gera dificuldade para automatizar loops por espécie, aumenta chance de erro e dificulta manutenção do projeto.

Por isso, este notebook organiza essa base inicial.

## O que ele faz

O notebook executa as seguintes etapas:

1. **Configurações de ambiente**  
   Imports para o desenvolvido, centralização de todos os caminhos do projeto.

2. **Inspeção dos arquivos brutos**  
   Lista os arquivos baixados e identifica quais são `cds` e `ncrna`.

3. **Descompactação dos arquivos (`.fa.gz`)**  
   Converte os arquivos para `.fa`, deixando-os utilizáveis nas próximas etapas.

4. **Normalização dos nomes**  
   Simplifica os nomes dos arquivos para um padrão consistente por espécie e tipo (cds / ncrna).

5. **Organização da estrutura**  
   Garante que os arquivos estejam organizados de forma previsível para uso em scripts futuros.

6. **Validação dos dados**  
   Verifica se cada espécie possui os dois grupos esperados:
   - codificante (`cds`)
   - não codificante (`ncrna`)

   Verifica estatísticas sobre as sequencias:
   - Quantidade de sequencias
   - Menor cadeia
   - Maior cadeia

## Resultado esperado

Ao final, os dados estarão:
- descompactados  
- com nomes padronizados  
- organizados por espécie  
- prontos para uso no restante do pipeline

## 1. Configurações de ambiente

### 1.1 Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

### 1.2 Paths

In [13]:
# Pasta base do projeto RNAmining
BASE_RNAMINING = Path("/home/samuel/projects/RNAmining-updated")

# Pasta com os novos FASTAs do ensemble
S5_DIR = BASE_RNAMINING / "volumes" / "rnamining-front" / "data" / "S5_File"

# Pasta onde os arquivos organizados serão armazenados
ORG_DIR = S5_DIR / "Organisms_Sequences"

In [14]:
# Cria a pasta caso não exista
ORG_DIR.mkdir(exist_ok=True)

### 2. Inspeção dos arquivos brutos

Checagem de quantos arquivos FASTAs compactados foram extraidos para a pasta

In [3]:
gz_files = list(S5_DIR.glob("*.gz"))

print(f"Total de arquivos .gz: {len(gz_files)}")

Total de arquivos .gz: 32


Dos arquivos, quantos são cds e quantos são ncRNA. Checado apenas para garantir que todas as espécies tiveram suas sequencias codificantes e não codificantes extraidas.

In [4]:
cds_files = [f for f in gz_files if "cds" in f.name]
ncrna_files = [f for f in gz_files if "ncrna" in f.name]

print(f"CDS: {len(cds_files)}")
print(f"ncRNA: {len(ncrna_files)}")

CDS: 16
ncRNA: 16


Exemplos dos arquivos extraidos.

In [5]:
print("CDS: ")
for f in cds_files[:5]:
    print(f.name)

print("\nncRNA: ")
for f in ncrna_files[:5]:
    print(f.name)

CDS: 
Notechis_scutatus.TS10Xv2-PRI.cds.all.fa.gz
Sphenodon_punctatus.ASM311381v1.cds.all.fa.gz
Xenopus_tropicalis.UCB_Xtro_10.0.cds.all.fa.gz
Danio_rerio.GRCz11.cds.all.fa.gz
Rattus_norvegicus.GRCr8.cds.all.fa.gz

ncRNA: 
Rattus_norvegicus.GRCr8.ncrna.fa.gz
Ornithorhynchus_anatinus.mOrnAna1.p.v1.ncrna.fa.gz
Crocodylus_porosus.CroPor_comp1.ncrna.fa.gz
Gallus_gallus.bGalGal1.mat.broiler.GRCg7b.ncrna.fa.gz
Homo_sapiens.GRCh38.ncrna.fa.gz


Ao inspecionar os nomes dos arquivos é possível identificar o possível padrão: Nome_da_especie.resto.(cds/ncrna).gz
A estratégia será salvar o nome de cada espécie, coletando o nome utilizando do padrão encontrado nos arquivos.

### 3. Descompactação dos arquivos (`.fa.gz`)

In [8]:
print(nomes_especies)
print(len(nomes_especies))

{'Monodelphis_domestica', 'Rattus_norvegicus', 'Xenopus_tropicalis', 'Sphenodon_punctatus', 'Crocodylus_porosus', 'Ornithorhynchus_anatinus', 'Mus_musculus', 'Danio_rerio', 'Petromyzon_marinus', 'Chrysemys_picta_bellii', 'Latimeria_chalumnae', 'Gallus_gallus', 'Anolis_carolinensis', 'Homo_sapiens', 'Notechis_scutatus', 'Eptatretus_burgeri'}
16


In [9]:
for file in S5_DIR.glob("*.fa.gz"):
    # Os arquivos extraidos são salvos com o seu nome sem o ".gz"
    output_file = file.with_suffix("")

    # Se ele já foi extraido, pula
    if output_file.exists():
        continue

    # Script do gunzip para descompactar
    !gunzip -c "{file}" > "{output_file}"
    

### 4. Normatização dos nomes

#### 4.1 Guardar os nomes de cada espécie

Será extraido o nome completo das espécies analisadas para a montagem de arquivos futuros, como: Nomes simplificados para facilitar a navegação nos dados, nomes de pastas .....

In [6]:
# Essa lista é um set, pois evita duplicatas ao lidar com um arquivo para cds e outro para ncrna sobre a mesma espécie
nomes_especies = set()

In [7]:
# Looping que percorre, no diretório S5_DIR, cada arquivo terminado com "".fa.gz"
for file in S5_DIR.glob("*.fa.gz"):
    # os nomes das espécies estão antes do primeiro ponto
    especies = file.name.split(".")[0]
    # Adiciona á lista
    nomes_especies.add(especies)

In [ ]:
for file in S5_DIR.glob("*.fa"):
    # Os nomes das espécies estão antes do primeiro ponto
    especies = file.name.split(".")[0]

    # Checa qual é o tipo do arquivo em questão e guarda essa informação 
    if "cds" in file.name:
        tipo = "cds"
    elif "ncrna" in file.name:
        tipo = "ncrna"

    # Monta o novo nome
    novo_nome = f"{especies}.{tipo}.fa"

    # Print apenas pra acompanhar o processo
    print(file.name, "->", novo_nome)


    # Novo arquivo com nome atualizado
    novo_path = file.parent / novo_nome

    if novo_path.exists():
        continue

    # Subistitui o arquivo antigo com o novo
    file.rename(novo_path)


Latimeria_chalumnae.LatCha1.ncrna.fa -> Latimeria_chalumnae.ncrna.fa
Rattus_norvegicus.GRCr8.cds.all.fa -> Rattus_norvegicus.cds.fa
Anolis_carolinensis.AnoCar2.0v2.cds.all.fa -> Anolis_carolinensis.cds.fa
Sphenodon_punctatus.ASM311381v1.ncrna.fa -> Sphenodon_punctatus.ncrna.fa
Danio_rerio.GRCz11.ncrna.fa -> Danio_rerio.ncrna.fa
Mus_musculus.GRCm39.cds.all.fa -> Mus_musculus.cds.fa
Xenopus_tropicalis.UCB_Xtro_10.0.ncrna.fa -> Xenopus_tropicalis.ncrna.fa
Eptatretus_burgeri.Eburgeri_3.2.cds.all.fa -> Eptatretus_burgeri.cds.fa
Sphenodon_punctatus.ASM311381v1.cds.all.fa -> Sphenodon_punctatus.cds.fa
Eptatretus_burgeri.Eburgeri_3.2.ncrna.fa -> Eptatretus_burgeri.ncrna.fa
Mus_musculus.GRCm39.ncrna.fa -> Mus_musculus.ncrna.fa
Ornithorhynchus_anatinus.mOrnAna1.p.v1.ncrna.fa -> Ornithorhynchus_anatinus.ncrna.fa
Chrysemys_picta_bellii.Chrysemys_picta_bellii-3.0.3.cds.all.fa -> Chrysemys_picta_bellii.cds.fa
Monodelphis_domestica.ASM229v1.ncrna.fa -> Monodelphis_domestica.ncrna.fa
Danio_rerio.GRCz1

### 5. Organização da estrutura

In [ ]:
# Looping que percorre todos os arquivos .fa
for file in S5_DIR.glob("*.fa"):
    
    # Define o novo caminho do arquivo dentro da nova pasta
    new_path = ORG_DIR / file.name
    
    if new_path.exists():
        continue

    # Move o arquivo para a pasta criada
    file.rename(new_path)
    
    # Print apenas para acompanhar
    print(file.name, "->", new_path)

Chrysemys_picta_bellii.ncrna.fa -> /home/samuel/projects/RNAmining-updated/volumes/rnamining-front/data/S5_File/Organisms_Sequences/Chrysemys_picta_bellii.ncrna.fa
Rattus_norvegicus.ncrna.fa -> /home/samuel/projects/RNAmining-updated/volumes/rnamining-front/data/S5_File/Organisms_Sequences/Rattus_norvegicus.ncrna.fa
Homo_sapiens.ncrna.fa -> /home/samuel/projects/RNAmining-updated/volumes/rnamining-front/data/S5_File/Organisms_Sequences/Homo_sapiens.ncrna.fa
Gallus_gallus.cds.fa -> /home/samuel/projects/RNAmining-updated/volumes/rnamining-front/data/S5_File/Organisms_Sequences/Gallus_gallus.cds.fa
Notechis_scutatus.cds.fa -> /home/samuel/projects/RNAmining-updated/volumes/rnamining-front/data/S5_File/Organisms_Sequences/Notechis_scutatus.cds.fa
Eptatretus_burgeri.ncrna.fa -> /home/samuel/projects/RNAmining-updated/volumes/rnamining-front/data/S5_File/Organisms_Sequences/Eptatretus_burgeri.ncrna.fa
Crocodylus_porosus.cds.fa -> /home/samuel/projects/RNAmining-updated/volumes/rnamining-fro

### 6. Validação dos dados